In [1]:
import numpy as np
import pandas as pd

In [2]:
np.random.seed(42)

X = pd.DataFrame({
    "age": np.random.randint(18, 65, 300),
    "income": np.random.normal(60000, 12000, 300),
    "experience": np.random.randint(0, 40, 300),
    "noise1": np.random.randn(300),
    "noise2": np.random.randn(300)
})

y = (X["income"] > 60000).astype(int)

### 1. Variance Threshold (Remove Useless Features)
Features with near-zero variance -> useless.

In [3]:
from sklearn.feature_selection import VarianceThreshold

vt = VarianceThreshold(threshold=0.01)
X_vt = vt.fit_transform(X)

vt.get_support()

array([ True,  True,  True,  True,  True])

- Removes constants
- First-pass cleanup

### 2. Correlation Threshold
Remove highly correlated features.

In [4]:
corr = X.corr().abs()

upper = corr.where(
    np.triu(np.ones(corr.shape), k=1).astype(bool)
)

to_drop = [col for col in upper.columns if any(upper[col] > 0.85)]
X.drop(columns=to_drop)

,age,income,experience,noise1,noise2
0,56,59941.716664,17,-0.718303,-1.846772
1,46,55400.120325,27,0.905888,-0.566669
2,32,54449.677678,21,-0.672468,1.163485
3,60,46650.031748,20,-0.655110,-0.929332
4,25,70406.753988,5,0.445399,-0.780302
...,...,...,...,...,...
295,52,72884.184145,16,0.877102,0.922932
296,33,60912.576479,23,0.463724,-1.868200
297,58,64235.519706,24,0.577817,-1.974137
298,53,67393.279848,6,-0.484309,1.042609


### 3. Chi-Square Test (Categorical -> Target)
- For non-negative features
- Classification only

In [5]:
from sklearn.feature_selection import chi2, SelectKBest

selector = SelectKBest(score_func=chi2, k=2)
X_new = selector.fit_transform(abs(X), y)

### 4. Anova F-Test (Numerical -> Target)
- Strong baseline for classification

In [6]:
from sklearn.feature_selection import f_classif

selector = SelectKBest(score_func=f_classif, k=3)
X_new = selector.fit_transform(X, y)

#### When to Use Filter Methods
- Very wide datasets
- Early screening
- Before expensive models

### Wrapper & Embedded Methods
Use models themseleves to choose features.

#### 1. Embedded: Lasso (L1 Regularization)

In [7]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(penalty="l1", solver="liblinear")
model.fit(X, y)

importance = pd.Series(model.coef_[0], index=X.columns)
importance

/Users/jatinrokde/CodeBase/SynapseWorks/MLForge/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/jatinrokde/CodeBase/SynapseWorks/MLForge/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


age          -0.016734
income        0.000432
experience   -0.015832
noise1       -0.017539
noise2        0.054713
dtype: float64

Coefficients -> 0 -> feature dropped.   
Great for linear models

#### 2. Tree-Based Feature Importance

In [8]:
from sklearn.ensemble import RandomForestClassifier

random_forest = RandomForestClassifier()
random_forest.fit(X, y)

pd.Series(random_forest.feature_importances_, index=X.columns)

age           0.008330
income        0.942429
experience    0.015602
noise1        0.013761
noise2        0.019878
dtype: float64

- Captures non-linearities
- Handles interactions

#### 3. Recursive Feature Elimination (RFE)

In [9]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

rfe = RFE(LogisticRegression(), n_features_to_select=3)
rfe.fit(X, y)

rfe.support_

/Users/jatinrokde/CodeBase/SynapseWorks/MLForge/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


array([False, False,  True,  True,  True])

- Accurate 
- Expensive